In [1]:
# pip install 명령어
!pip install torch transformers sentencepiece sentence-transformers qdrant-client langchain langchain-community langchain-openai openai scikit-learn rank_bm25 tiktoken numpy tqdm langchain-qdrant


In [2]:
import os
from typing import List, Dict, Any, Optional, Tuple
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import ScoredPoint
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.qdrant import Qdrant as LangchainQdrant
from langchain.schema import Document
from langchain_openai import OpenAI
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
import logging
import time
import numpy as np
from openai import OpenAI as OpenAIClient
from google.colab import userdata
import difflib

In [3]:
# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# OpenAI API 키 설정
openai_api_key = userdata.get('FINAL_TEAM3')

# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
COLLECTION_NAME = "son99_d"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# 임베딩 모델 초기화
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")


/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Langchain의 page_content 필드값에 Qdrant의 답변 값이 들어가도록 설정
 - payload_key로 되어 있는데, 이는 아래 초기화 쪽에서 '답변'이라고 직접적으로 들어감.

In [4]:
class CustomQdrant(LangchainQdrant):
    def _document_from_scored_point(
        self,
        scored_point: ScoredPoint,
        content_payload_key: str,
        metadata_payload_key: str,
        score_key: str,
    ) -> Document:
        payload = scored_point.payload or {}
        content = payload.get(content_payload_key, '')
        metadata = payload.get(metadata_payload_key, {})
        metadata[score_key] = scored_point.score
        return Document(page_content=content, metadata=metadata)

In [5]:
# Langchain Qdrant 초기화
vector_store = CustomQdrant(
    client=client,
    collection_name=COLLECTION_NAME,
    embeddings=embeddings,
    content_payload_key='답변'
)

# Sentence Transformer 모델
st_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# OpenAI 클라이언트 초기화
openai_client = OpenAIClient(api_key=openai_api_key)

# GPT 사용 횟수 추적을 위한 전역 변수
gpt_usage_count = 0
MAX_GPT_USAGE = 3

# 샘플 모델 초기화
model_name = "centwon/ko-gpt-trinity-cw"
tokenizer = AutoTokenizer.from_pretrained(model_name)


메타데이터 가져오는 부분
 - 기본값으로 일반, 미분류, 정보요청 값을 넣어둠.

In [6]:
def get_unique_metadata():
    default_categories = ["일반"]
    default_diseases = ["미분류"]
    default_intents = ["정보요청"]

    try:
        points = client.scroll(
            collection_name=COLLECTION_NAME,
            limit=10000,
            with_payload=True
        )[0]

        if not points:
            logging.warning("데이터베이스에서 포인트를 찾을 수 없습니다. 기본값을 사용합니다.")
            return default_categories, default_diseases, default_intents

        categories, diseases, intents = set(), set(), set()

        for point in points:
            if not point.payload:
                continue
            metadata = point.payload.get('metadata', {})
            if metadata.get('카테고리'):
                categories.add(metadata['카테고리'])
            if metadata.get('질병'):
                diseases.add(metadata['질병'])
            if metadata.get('의도'):
                intents.add(metadata['의도'])

        categories = sorted(filter(None, categories)) or default_categories
        diseases = sorted(filter(None, diseases)) or default_diseases
        intents = sorted(filter(None, intents)) or default_intents

        logging.info(f"메타데이터 추출 완료: {len(categories)} 카테고리, {len(diseases)} 질병, {len(intents)} 의도")
        return categories, diseases, intents

    except Exception as e:
        logging.error(f"메타데이터 추출 중 오류 발생: {str(e)}", exc_info=True)
        logging.warning("오류로 인해 기본 메타데이터를 사용합니다.")
        return default_categories, default_diseases, default_intents


In [7]:
def match_metadata(query: str, categories: List[str], diseases: List[str], intents: List[str]) -> Tuple[str, str, str]:
    def find_best_match(text: str, options: List[str]) -> str:
        matches = difflib.get_close_matches(text.lower(), [opt.lower() for opt in options], n=1, cutoff=0.6)
        return options[[opt.lower() for opt in options].index(matches[0])] if matches else ""

    words = query.lower().split()
    category = next((find_best_match(word, categories) for word in words if find_best_match(word, categories)), "일반")
    disease = next((find_best_match(word, diseases) for word in words if find_best_match(word, diseases)), "미분류")

    # 의도 추출 로직 개선
    intent_keywords = {
        "정보요청": ["무엇", "어떤", "알려줘", "설명해줘"],
        "예방": ["예방", "방지", "막다"],
        "치료": ["치료", "낫다", "해결"],
        "증상": ["증상", "징후", "나타나다"]
    }

    intent = "정보요청"
    for intent_type, keywords in intent_keywords.items():
        if any(keyword in query.lower() for keyword in keywords):
            intent = intent_type
            break

    return category, disease, intent

In [8]:
def expand_query(query: str) -> List[str]:
    synonyms = {
        "두통": ["머리 아픔", "머리痛"],
        "감기": ["콧물", "기침"],
        "소화불량": ["속 쓰림", "더부룩함"],
        "증상": ["징후", "병증", "증세"],
        "치료": ["처치", "요법", "치료법"],
        "예방": ["방지", "예방책", "예방법"],
        "통증": ["아픔", "고통", "불편함"],
    }

    expanded_queries = [query]
    for word, syns in synonyms.items():
        if word in query:
            for syn in syns:
                expanded_queries.append(query.replace(word, syn))

    return list(set(expanded_queries))

여기 함수 없음

In [9]:
def bm25_search(query: str, documents: List[Any], top_k: int = 5) -> List[Dict[str, Any]]:
    if not documents:
        logging.warning("BM25 검색: 문서 목록이 비어 있습니다.")
        return [{"error": "문서 목록이 비어 있습니다."}]

    try:
        corpus = [doc.payload.get('답변', '') for doc in documents if doc.payload.get('답변')]
        if not corpus:
            logging.warning("BM25 검색: 모든 문서의 내용이 비어 있습니다.")
            return [{"error": "유효한 문서 내용이 없습니다."}]

        bm25 = BM25Okapi(corpus)
        scores = bm25.get_scores(query.split())

        top_results = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)[:top_k]

        results = []
        for doc, score in top_results:
            if doc.payload.get('답변'):
                results.append({
                    "content": doc.payload['답변'],
                    "score": score,
                    "metadata": doc.payload.get('metadata', {})
                })

        return results if results else [{"error": "검색 결과가 없습니다."}]
    except Exception as e:
        logging.error(f"BM25 검색 중 오류 발생: {str(e)}", exc_info=True)
        return [{"error": f"검색 중 오류 발생: {str(e)}"}]


In [10]:
def vector_search(query: str, documents: List[Any], top_k: int = 5) -> List[Dict[str, Any]]:
    if not documents:
        logging.warning("벡터 검색: 문서 목록이 비어 있습니다.")
        return [{"error": "문서 목록이 비어 있습니다."}]

    try:
        query_embedding = st_model.encode([query])[0]
        doc_contents = [doc.payload.get('답변', '') for doc in documents if doc.payload.get('답변')]

        if not doc_contents:
            logging.warning("벡터 검색: 유효한 문서 내용이 없습니다.")
            return [{"error": "유효한 문서 내용이 없습니다."}]

        doc_embeddings = st_model.encode(doc_contents)

        similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

        top_results = sorted(zip(documents, similarities), key=lambda x: x[1], reverse=True)[:top_k]

        results = []
        for doc, score in top_results:
            if doc.payload.get('답변', '').strip():
                results.append({
                    "content": doc.payload['답변'],
                    "score": float(score),
                    "metadata": doc.payload.get('metadata', {})
                })

        return results if results else [{"error": "검색 결과가 없습니다."}]
    except Exception as e:
        logging.error(f"벡터 검색 중 오류 발생: {str(e)}", exc_info=True)
        return [{"error": f"검색 중 오류 발생: {str(e)}"}]


In [11]:
def ensemble_search(query: str, documents: List[Any], top_k: int = 5) -> List[Dict[str, Any]]:
    if not documents:
        logging.warning("앙상블 검색: 문서 목록이 비어 있습니다.")
        return [{"error": "문서 목록이 비어 있습니다."}]

    try:
        bm25_results = bm25_search(query, documents, top_k)
        vector_results = vector_search(query, documents, top_k)

        if "error" in bm25_results[0] and "error" in vector_results[0]:
            return [{"error": "BM25와 벡터 검색 모두 결과가 없습니다."}]

        bm25_results = [r for r in bm25_results if "error" not in r]
        vector_results = [r for r in vector_results if "error" not in r]

        combined_results = {}
        for result in bm25_results + vector_results:
            content = result["content"]
            if content not in combined_results or result["score"] > combined_results[content]["score"]:
                combined_results[content] = result

        final_results = sorted(combined_results.values(), key=lambda x: x["score"], reverse=True)[:top_k]

        return final_results if final_results else [{"error": "유효한 검색 결과가 없습니다."}]
    except Exception as e:
        logging.error(f"앙상블 검색 중 오류 발생: {str(e)}", exc_info=True)
        return [{"error": f"검색 중 오류 발생: {str(e)}"}]


여기까지

In [12]:
def generate_custom_model_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> str:
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name).to('cuda' if torch.cuda.is_available() else 'cpu')

        prompt = f"질문: {query}\n\n컨텍스트: {context or ''}\n\n메타데이터: {metadata or {}}\n\n답변:"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        max_length = min(150 + len(query) * 2, 500)

        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            temperature=0.7,
            do_sample=True
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response.split("답변:")[-1].strip()
    except Exception as e:
        logging.error(f"커스텀 모델 응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return f"죄송합니다. 응답을 생성하는 중에 오류가 발생했습니다: {str(e)}"
    finally:
        if 'model' in locals():
            del model
            torch.cuda.empty_cache()

In [13]:
def generate_gpt4_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> str:
    if not openai_api_key:
        logging.error("OpenAI API 키가 설정되지 않았습니다.")
        return "OpenAI API 키가 설정되지 않아 GPT-4를 사용할 수 없습니다."

    try:
        system_message = """
        당신은 의료 정보를 제공하는 전문가 어시스턴트입니다. 주어진 컨텍스트와 메타데이터를 바탕으로
        정확하고 간결한 답변을 제공해야 합니다. 답변은 완전한 문장으로 끝내야 하며, 중간에 끊기지 않아야 합니다.
        답변의 길이는 약 1000자 내외로 제한해 주세요.
        """
        user_message = f"질문: {query}\n\n컨텍스트: {context or '관련 컨텍스트 없음'}\n\n메타데이터: {metadata or '메타데이터 없음'}"

        logging.info(f"GPT-4에 요청 보내는 중: 질문='{query}'")

        response = openai_client.chat.completions.create(
            model="gpt-4-turbo-preview",
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": user_message}
            ],
            max_tokens=1000,
            n=1,
            stop=None,
            temperature=0.7,
        )

        generated_response = response.choices[0].message.content.strip()
        logging.info(f"GPT-4 응답 생성 완료: 길이={len(generated_response)}")

        return generated_response

    except Exception as e:
        logging.error(f"GPT-4 응답 생성 중 오류 발생: {str(e)}", exc_info=True)
        return f"죄송합니다. GPT-4 응답을 생성하는 중에 오류가 발생했습니다: {str(e)}"

In [14]:
def generate_response(query: str, context: Optional[str] = None, metadata: Optional[Dict[str, str]] = None) -> str:
    global gpt_usage_count

    if gpt_usage_count < MAX_GPT_USAGE:
        gpt_usage_count += 1
        response = generate_gpt4_response(query, context, metadata)
        logging.info(f"GPT-4 사용 (남은 횟수: {MAX_GPT_USAGE - gpt_usage_count})")
    else:
        response = generate_custom_model_response(query, context, metadata)
        logging.info("커스텀 모델 사용")

    if len(response) < 10 or "오류" in response:
        logging.warning(f"생성된 응답이 너무 짧거나 오류를 포함: {response}")
        response = "죄송합니다. 적절한 응답을 생성하지 못했습니다. 다시 질문해 주시겠어요?"

    return response


In [15]:
def process_query(query: str) -> str:
    start_time = time.time()
    logging.info(f"처리 중인 쿼리: {query}")

    try:
        category, disease, intent = match_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS)
        logging.info(f"추출된 메타데이터: 카테고리={category}, 질병={disease}, 의도={intent}")
        print(f"추출된 메타데이터: 카테고리={category}, 질병={disease}, 의도={intent}")

        filter_condition = {
            "should": [
                {"key": "metadata.카테고리", "match": {"value": category}},
                {"key": "metadata.질병", "match": {"value": disease}},
                {"key": "metadata.의도", "match": {"value": intent}}
            ]
        }

        all_documents = vector_store.similarity_search_with_score(
            query, k=5, filter=filter_condition
        )

        if not all_documents:
            logging.warning("Qdrant에서 검색 결과가 없습니다. 필터 없이 재검색합니다.")
            all_documents = vector_store.similarity_search_with_score(query, k=5)

        if not all_documents:
            logging.warning("Qdrant에서 검색 결과가 없습니다.")
            context = None
        else:
            context = "\n".join([doc.page_content for doc, _ in all_documents])

        metadata = {"카테고리": category, "질병": disease, "의도": intent}
        if all_documents:
            metadata.update(all_documents[0][0].metadata)

        response = generate_response(query, context=context, metadata=metadata)

        end_time = time.time()
        processing_time = end_time - start_time
        logging.info(f"처리 시간: {processing_time:.2f}초")

        return response

    except Exception as e:
        logging.error(f"쿼리 처리 중 오류 발생: {str(e)}", exc_info=True)
        return generate_response(query, context=None, metadata={"카테고리": category, "질병": disease, "의도": intent})

In [16]:
def evaluate_system() -> None:
    global gpt_usage_count
    gpt_usage_count = 0  # 평가 시작 전 카운터 초기화

    test_queries = [
        "고혈압의 주요 증상은 무엇인가요?",
        "당뇨병 예방을 위한 식습관 조언 부탁드립니다.",
        "감기와 독감의 차이점은 무엇인가요?",
        "코로나19 백신의 부작용에 대해 알고 싶어요.",
        "불면증 해소를 위한 방법을 알려주세요.",
        "알레르기 비염의 원인과 치료법은?",
        "어린이 비만 예방을 위한 조언이 필요합니다.",
        "갑상선 기능 저하증의 증상은 어떤 것들이 있나요?",
        "만성 피로 증후군의 원인과 대처 방법은?",
        "치매 초기 증상을 어떻게 알 수 있나요?"
    ]

    total_time = 0
    successful_queries = 0
    failed_queries = 0
    results = []

    print("시스템 평가를 시작합니다...")
    print("=" * 50)

    for i, query in enumerate(test_queries, 1):
        print(f"\n테스트 쿼리 {i}: {query}")
        print("-" * 50)

        try:
            start_time = time.time()
            print(f"쿼리 처리 시작: {query}")
            response = process_query(query)
            end_time = time.time()

            processing_time = end_time - start_time
            total_time += processing_time

            print(f"응답: {response}")
            print(f"처리 시간: {processing_time:.2f}초")
            print(f"사용 모델: {'GPT-4' if gpt_usage_count <= MAX_GPT_USAGE else '커스텀 모델'}")

            successful_queries += 1
            results.append({
                "query": query,
                "response": response,
                "processing_time": processing_time,
                "model_used": 'GPT-4' if gpt_usage_count <= MAX_GPT_USAGE else '커스텀 모델'
            })

        except Exception as e:
            logging.error(f"쿼리 처리 중 오류 발생: {str(e)}", exc_info=True)
            print(f"오류 발생: {str(e)}")
            failed_queries += 1
            results.append({
                "query": query,
                "error": str(e),
                "processing_time": None,
                "model_used": None
            })

        print("-" * 50)

    print("\n평가 결과 요약:")
    print("=" * 50)
    print(f"총 쿼리 수: {len(test_queries)}")
    print(f"성공한 쿼리 수: {successful_queries}")
    print(f"실패한 쿼리 수: {failed_queries}")

    if successful_queries > 0:
        avg_time = total_time / successful_queries
        print(f"평균 처리 시간: {avg_time:.2f}초")
    else:
        print("성공한 쿼리가 없어 평균 처리 시간을 계산할 수 없습니다.")

    print(f"GPT-4 사용 횟수: {min(gpt_usage_count, MAX_GPT_USAGE)}")
    print(f"커스텀 모델 사용 횟수: {max(0, successful_queries - MAX_GPT_USAGE)}")
    print("=" * 50)

In [17]:
def run_system():
    print("의료 정보 시스템을 초기화하는 중...")

    # 메타데이터 초기화
    global DISEASE_CATEGORIES, DISEASES, INTENTS
    DISEASE_CATEGORIES, DISEASES, INTENTS = get_unique_metadata()
    print(f"메타데이터 로드 완료: {len(DISEASE_CATEGORIES)} 카테고리, {len(DISEASES)} 질병, {len(INTENTS)} 의도")

    # Qdrant 연결 확인
    try:
        client.get_collections()
        print("Qdrant 서버에 연결됨")
    except Exception as e:
        print(f"Qdrant 서버 연결 실패: {str(e)}")
        return

    print("\n의료 정보 시스템이 준비되었습니다. 질문을 입력하세요. ('종료'를 입력하면 프로그램이 종료됩니다)")

    while True:
        query = input("\n질문: ").strip()
        if query.lower() == '종료':
            print("프로그램을 종료합니다.")
            break

        try:
            response = process_query(query)
            print(f"\n답변: {response}")
        except Exception as e:
            print(f"오류 발생: {str(e)}")
            logging.error(f"쿼리 처리 중 오류 발생: {str(e)}", exc_info=True)

if __name__ == "__main__":
    try:
        run_system()
    except Exception as e:
        print(f"프로그램 실행 중 예기치 않은 오류 발생: {str(e)}")
        logging.exception("예외 발생")

의료 정보 시스템을 초기화하는 중...
메타데이터 로드 완료: 1 카테고리, 1 질병, 1 의도
Qdrant 서버에 연결됨

의료 정보 시스템이 준비되었습니다. 질문을 입력하세요. ('종료'를 입력하면 프로그램이 종료됩니다)

질문: 응급한 고혈압 증상이 궁금해요
추출된 메타데이터: 카테고리=일반, 질병=미분류, 의도=증상


ERROR:root:쿼리 처리 중 오류 발생: 'str' object does not support item assignment
Traceback (most recent call last):
  File "<ipython-input-15-cdf4d8db14a7>", line 24, in process_query
    all_documents = vector_store.similarity_search_with_score(query, k=5)
  File "/usr/local/lib/python3.10/dist-packages/langchain_community/vectorstores/qdrant.py", line 366, in similarity_search_with_score
    return self.similarity_search_with_score_by_vector(
  File "/usr/local/lib/python3.10/dist-packages/langchain_community/vectorstores/qdrant.py", line 625, in similarity_search_with_score_by_vector
    return [
  File "/usr/local/lib/python3.10/dist-packages/langchain_community/vectorstores/qdrant.py", line 627, in <listcomp>
    self._document_from_scored_point(
  File "<ipython-input-4-c5d599e6fb71>", line 12, in _document_from_scored_point
    metadata[score_key] = scored_point.score
TypeError: 'str' object does not support item assignment



답변: 응급한 고혈압 증상은 매우 심각한 상태로, 즉각적인 의료 조치가 필요합니다. 이러한 상태는 고혈압 위기(hypertensive crisis)로 분류되며, 혈압이 매우 높아 심장, 뇌, 신장 및 기타 장기에 손상을 줄 위험이 있습니다. 응급한 고혈압의 증상으로는 다음과 같은 것들이 포함됩니다:

1. 심한 두통: 갑작스럽고, 심한 두통이 발생할 수 있으며, 이는 흔히 고혈압 위기의 초기 증상 중 하나입니다.
2. 시력 문제: 시력이 흐려지거나, 시야에 변화가 생기는 등의 시력 문제가 발생할 수 있습니다.
3. 가슴 통증: 가슴에 통증이 느껴지거나, 가슴이 불편하게 느껴질 수 있습니다. 때로는 심장 부근에 압박감이나 무거운 느낌이 들기도 합니다.
4. 호흡 곤란: 숨쉬기가 어려워지거나, 답답함을 느낄 수 있습니다.
5. 코피: 갑작스러운 코피가 발생할 수 있으며, 이는 혈압이 상승했음을 나타낼 수 있습니다.
6. 심한 뒷목의 통증: 뒷목에 심한 통증이 발생할 수 있으며, 이는 특히 누웠을 때 더욱 느껴질 수 있습니다.
7. 메스꺼움 또는 구토: 고혈압이 심각한 경우, 메스꺼움이나 구토 증상이 동반될 수 있습니다.
8. 발작: 매우 드물지만, 고혈압 위기가 심각한 경우 발작이 발생할 수 있습니다.
9. 의식 혼란 또는 변화: 심각한 고혈압은 뇌에 영향을 줄 수 있어, 환자가 혼란스러워하거나 의식에 변화가 생길 수 있습니다.

이러한 증상 중 하나 이상을 경험한다면, 즉시 의료 기관을 방문하거나 응급 구조 요청을 해야 합니다. 고혈압 위기는 즉각적인 치료를 받지 않을 경우 장기 손상이나 생명을 위협할 수 있는 다른 심각한 건강 문제로 이어질 수 있습니다.

질문: 종료
프로그램을 종료합니다.


이 코드에서의 문제
 - 메타데이터 : 기본값으로 설정한 부분만 나옴
 - DB에서 검색 결과를 못 찾음
 - 쿼리문 처리 중 오류 => CustomQdrant 클래스의 _document_from_scored_point 메서드에서 문제가 있는 듯 함.

이후 해결 방안 :
1. DB만 가져와서 메타데이터 필터링이 되는지 확인
2. BM25 와 앙상블 검색기 성능 확인
3. DB에서 컨택스트를 기반으로 무료 모델의 답변 생성 확인
4. gpt-4o-turbo 연결
5. langchain 연결
6. 동의어 처리
7. 전체 성능 확인